# 🧠 Model Explainability, SHAP & Business Financial ROI

## Objectives:
1. **Technical Rigor**: Prove model generalization with comprehensive test set metrics (PR-AUC, ROC-AUC, Brier Score, Cost-Weighted F1).
2. **Global Explainability**: Quantify overall feature importance using SHAP (Shapley Additive exPlanations).
3. **Local Explainability & Reason Codes**: Deconstruct individual customer risk predictions into interpretable positive and negative drivers.
4. **Prescriptive Action Playbook**: Map machine learning risk factors to targeted business retention workflows.
5. **Financial Decision Intelligence**: Apply the Expected Value Framework to simulate net revenue saved and optimize decision thresholds.

In [1]:
import os, sys
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(project_root)
sys.path.insert(0, str(project_root))

from src.evaluation.explainability import ChurnExplainer, optimize_business_threshold
from src.models.registry import ModelRegistry
print('Environment and Explainability suite initialized.')

In [2]:
# 1. Technical Evaluation & Model Quality
with open('models/churn_metadata.json', 'r') as f:
    churn_meta = json.load(f)

print(f'Production Model: {churn_meta["best_model_name"]} (Calibrated: {churn_meta["is_calibrated"]})')
test_m = churn_meta['test_metrics']
display(pd.DataFrame([
    {'Metric': 'ROC-AUC', 'Score': f"{test_m['roc_auc']:.4f}", 'Interpretation': 'Strong discrimination between churners & loyalists'},
    {'Metric': 'PR-AUC', 'Score': f"{test_m['pr_auc']:.4f}", 'Interpretation': 'High precision across rare positive churn classes'},
    {'Metric': 'Brier Score Loss', 'Score': f"{test_m['brier_score']:.4f}", 'Interpretation': 'Well-calibrated probabilities matching empirical risk'},
    {'Metric': 'F1-Score', 'Score': f"{test_m['f1']:.4f}", 'Interpretation': 'Balanced precision and recall under cost matrix'},
    {'Metric': 'Accuracy', 'Score': f"{test_m['accuracy']*100:.1f}%", 'Interpretation': 'Standard accuracy on 15% holdout test set'}
]))

In [3]:
# 2. Global SHAP Feature Importance
explainer = ChurnExplainer()
# Display pre-generated global SHAP summary
from IPython.display import Image
Image(filename='reports/figures/07_shap_summary.png')

In [4]:
# 3. Local Customer Waterfall Attribution & Reason Codes
sample_high_risk_customer = {
    'tenure': 2,
    'monthly_charges': 92.50,
    'total_charges': 185.0,
    'charges_tenure_ratio': 61.67,
    'contract_Month-to-month': 1,
    'internet_service_Fiber optic': 1,
    'payment_method_Electronic check': 1,
    'security_bundle_score': 0
}

explanation = explainer.explain_customer(sample_high_risk_customer, clv=1250.0)

print(f"Customer Assessment: {explanation['risk_tier']} (Probability: {explanation['churn_probability']*100:.1f}%)")
print('\nTop Churn Risk Drivers:')
for d in explanation['top_risk_drivers']:
    print(f"  [+] {d['readable_name']}: +{d['shap_impact']:.3f} SHAP impact")

print('\nPrescribed Business Retention Actions:')
for act in explanation['prescribed_actions']:
    print(f"  -> Action: {act['action']}")
    print(f"     Detail: {act['detail']}")
    print(f"     Targeted Driver: {act['targeted_driver']} | Est. Cost: ${act['cost']:.2f}")

print('\nFinancial ROI Simulation for this Account:')
for k, v in explanation['financial_roi'].items():
    print(f"  {k}: {v}")

In [5]:
# 4. Display Local Waterfall Chart
Image(filename='reports/figures/08_shap_local_waterfall.png')

## Summary of Business Decision Intelligence
- **Transparency**: Non-technical account managers receive human-readable drivers rather than black-box probabilities.
- **Actionability**: Every identified risk driver links to an operational playbook with quantified execution costs.
- **Expected ROI**: Demonstrates clear financial surplus, proving that targeted retention delivers measurable ROI.